# 실습 2. Haystack — 컴포넌트 파이프라인

## 실습 목표

Haystack은 각 단계(임베더·retriever·프롬프트빌더·생성기)를 **컴포넌트**로 만들고
`connect`로 잇습니다(교안 2장). "흐름이 보인다"가 특징입니다.

- add_component + connect 로 파이프라인을 명시적으로 조립
- YAML 직렬화로 서빙도 가능(운영 친화)
- 같은 코퍼스·같은 질의로 6교시 비교

> `MLAPI_*` 필요. 임베딩 모델은 최초 1회 다운로드(캐시).

## 1. 컴포넌트 파이프라인 (교안 2.3)

문서를 임베딩해 저장한 뒤, 임베더→retriever→프롬프트→LLM을 `connect`로 잇습니다.

In [ ]:
from _common import (SAMPLE_DOCS, EVAL_QUESTION, MLAPI_BASE_URL, MLAPI_API_KEY,
                     MLAPI_MODEL, EMB_MODEL, have_mlapi)

PROMPT = """다음 문맥에만 근거해 한국어로 간결히 답하라. 문맥에 없으면 '찾을 수 없습니다'.
문맥:
{% for doc in documents %}- ({{ doc.meta.topic }}) {{ doc.content }}
{% endfor %}
질문: {{ question }}
답변:"""

def build_pipeline(top_k=3):
    from haystack import Pipeline, Document
    from haystack.document_stores.in_memory import InMemoryDocumentStore
    from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
    from haystack.components.embedders import (SentenceTransformersTextEmbedder,
                                               SentenceTransformersDocumentEmbedder)
    from haystack.components.builders import PromptBuilder
    from haystack.components.generators import OpenAIGenerator
    from haystack.utils import Secret
    store = InMemoryDocumentStore()
    docs = [Document(content=d["text"], meta={"topic": d["topic"]}) for d in SAMPLE_DOCS]
    doc_emb = SentenceTransformersDocumentEmbedder(model=EMB_MODEL); doc_emb.warm_up()
    store.write_documents(doc_emb.run(docs)["documents"])       # 임베딩 후 저장
    pipe = Pipeline()
    pipe.add_component("text_embedder", SentenceTransformersTextEmbedder(model=EMB_MODEL))
    pipe.add_component("retriever", InMemoryEmbeddingRetriever(store, top_k=top_k))
    pipe.add_component("prompt", PromptBuilder(template=PROMPT, required_variables=["question","documents"]))
    pipe.add_component("llm", OpenAIGenerator(api_key=Secret.from_token(MLAPI_API_KEY),
                                              api_base_url=MLAPI_BASE_URL, model=MLAPI_MODEL,
                                              generation_kwargs={"temperature": 0.2}))
    pipe.connect("text_embedder.embedding", "retriever.query_embedding")
    pipe.connect("retriever.documents", "prompt.documents")
    pipe.connect("prompt.prompt", "llm.prompt")
    return pipe

print("준비 완료")

## 2. 질의 실행 (교안 2.3 실측)

In [ ]:
pipe = build_pipeline(top_k=3)
q = EVAL_QUESTION
res = pipe.run({"text_embedder": {"text": q}, "prompt": {"question": q}},
               include_outputs_from={"retriever"})
print(f"질문: {q}\n")
print("답변:", res["llm"]["replies"][0].strip()[:200])
print("출처(retriever):", [d.meta.get("topic","?") for d in res["retriever"]["documents"]])

**포인트**

- add_component/connect 로 흐름이 코드에 그대로 보인다(가시성↑)
- LlamaIndex와 같은 출처·같은 의미의 답 → 검색 품질은 구성요소(임베딩)가 결정
- YAML 직렬화로 코드 없이 배포·서빙 가능

## 3. 하이브리드 + 리랭커 — `DocumentJoiner` 뒤에 `TransformersSimilarityRanker` connect

단일 임베딩 retriever를 넘어, **BM25(sparse) + 임베딩(dense)** 두 retriever 결과를 `DocumentJoiner`로
합치고(하이브리드), 그 뒤에 **`TransformersSimilarityRanker`** 를 connect해 (질문,문서)를 함께 보고
**정밀 재정렬**(precision↑)합니다. 리랭커는 한국어 정답을 위해 다국어 `BAAI/bge-reranker-v2-m3`를 씁니다(Day21 5교시 교훈).

```
te → emb_ret ↘
              joiner → ranker → (상위 문서)
   bm25_ret  ↗
```
> ⚠️ 최초 1회 `bge-reranker-v2-m3`(~2.3GB) 다운로드. `TransformersSimilarityRanker`는 legacy이며 후속은 `SentenceTransformersSimilarityRanker`.

In [ ]:
from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever, InMemoryBM25Retriever
from haystack.components.embedders import (SentenceTransformersTextEmbedder,
                                           SentenceTransformersDocumentEmbedder)
from haystack.components.joiners import DocumentJoiner
from haystack.components.rankers import TransformersSimilarityRanker

def build_hybrid_ranker_pipeline(top_k=5, top_r=3):
    store = InMemoryDocumentStore()
    de = SentenceTransformersDocumentEmbedder(model=EMB_MODEL); de.warm_up()
    store.write_documents(de.run([Document(content=d["text"], meta={"topic": d["topic"]})
                                  for d in SAMPLE_DOCS])["documents"])
    p = Pipeline()
    p.add_component("te", SentenceTransformersTextEmbedder(model=EMB_MODEL))
    p.add_component("emb_ret", InMemoryEmbeddingRetriever(store, top_k=top_k))    # dense
    p.add_component("bm25_ret", InMemoryBM25Retriever(store, top_k=top_k))        # sparse
    p.add_component("joiner", DocumentJoiner())                                   # 두 결과 병합
    p.add_component("ranker", TransformersSimilarityRanker(model="BAAI/bge-reranker-v2-m3", top_k=top_r))
    p.connect("te.embedding", "emb_ret.query_embedding")
    p.connect("emb_ret.documents", "joiner.documents")     # dense → joiner
    p.connect("bm25_ret.documents", "joiner.documents")    # sparse → joiner
    p.connect("joiner.documents", "ranker.documents")      # joiner → ranker (핵심)
    return p

hp = build_hybrid_ranker_pipeline()
q = EVAL_QUESTION
res = hp.run({"te": {"text": q}, "bm25_ret": {"query": q}, "ranker": {"query": q}},
             include_outputs_from={"joiner"})
print(f"질문: {q}")
print("joiner 병합 문서수:", len(res["joiner"]["documents"]), "(BM25+임베딩 중복 제거)")
print("ranker 재정렬 top3 :", [d.meta.get("topic","?") for d in res["ranker"]["documents"]])

**포인트 — joiner 뒤 ranker**

- `DocumentJoiner`가 BM25(sparse)+임베딩(dense) 결과를 **합쳐**(하이브리드, recall↑) 병합
- 그 뒤 `TransformersSimilarityRanker`가 (질문,문서)를 함께 보고 **정밀 재정렬**(precision↑) → top-3
- 컴포넌트를 **connect로 끼워넣는 것만으로** 2단계(검색→리랭크)가 완성 — Haystack의 조립식 강점
- 한국어엔 **다국어 리랭커**(bge) 필수(영어 ms-marco는 한국어 정답을 밀어냄, Day21 5교시)

## 4. `pipeline.dump()` — 파이프라인을 YAML로 직렬화

Haystack의 운영 강점: 파이프라인 구조를 **YAML로 저장**해 코드 없이 공유·버전관리·배포할 수 있습니다.
`pipeline.dump(file)`로 YAML 파일을 만들고, `Pipeline.load(file)`로 복원합니다.

In [ ]:
import os

YAML_PATH = "haystack_pipeline.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    hp.dump(f)                       # 파이프라인 → YAML 파일 생성

print(f"YAML 생성됨: {YAML_PATH}  ({os.path.getsize(YAML_PATH)} bytes)\n")
print("---- YAML 앞부분 ----")
print("\n".join(open(YAML_PATH, encoding="utf-8").read().splitlines()[:16]))

# 복원: YAML 파일만으로 동일 파이프라인 재구성
with open(YAML_PATH, encoding="utf-8") as f:
    reloaded = Pipeline.load(f)
print("\n복원된 컴포넌트:", sorted(reloaded.graph.nodes))

**포인트 — YAML 직렬화(운영 친화)**

- `dump()`로 만든 YAML은 **컴포넌트·연결·설정**을 모두 담아 코드 없이 배포·서빙·버전관리 가능
- `Pipeline.load()`로 그대로 복원 → 동일 파이프라인 재현(재현성↑)
- 검색/QA를 **제품으로 운영**할 때 LangChain·LlamaIndex 대비 Haystack의 차별점

# [TODO]

## [TODO] 1. 다른 질문으로 실행

`pipe.run(...)` 에 **다른 질문**을 넣어 답변·출처를 확인하세요.
(질문은 text_embedder와 prompt **둘 다**에 같은 값으로 넣어야 합니다.)

In [ ]:
# [TODO] 1: 다른 질문으로 실행
# [TODO] 여기에 코드를 작성하세요.


## [TODO] 2. top_k=5 파이프라인

`build_pipeline(top_k=5)` 로 새 파이프라인을 만들어 출처가 5개로 늘어나는지 확인하세요.

In [ ]:
# [TODO] 2: top_k=5 파이프라인
# [TODO] 여기에 코드를 작성하세요.


## 실습 정리

- Haystack = 명시적 컴포넌트 파이프라인(connect), 운영·서빙 친화
- 같은 임베딩이면 검색 결과는 LlamaIndex와 유사 — 차이는 코드 구조
- YAML 직렬화로 배포 용이